<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, VBox, HBox, FileUpload, Output, Label
from IPython.display import display, Markdown, HTML, clear_output
import io

# === 1. Oppsett for filopplasting ===
uploader = FileUpload(accept='', multiple=False, description="Last opp data")
app_display = Output()

def start_analysen(change):
    with app_display:
        clear_output(wait=True)
        if not uploader.value: return

        # --- ROBUST FILHENTING (Beholdt nøyaktig som sist) ---
        try:
            raw = uploader.value
            if isinstance(raw, (list, tuple)):
                file_info = raw[0]
            elif isinstance(raw, dict):
                file_info = list(raw.values())[0]
            else:
                file_info = raw

            content = file_info['content']
            df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')
            tid_data, niva_data = df.iloc[:,0].values, df.iloc[:,1].values
        except Exception as e:
            print(f"Feil ved lesing av fil: {e}")
            return

        # === 2. ESTIMERING (Beholdt nøyaktig som sist) ===
        y0_v = niva_data[0]
        A_v = niva_data[-1] - y0_v
        t10 = tid_data[np.where(niva_data > y0_v + 0.10 * A_v)[0][0]] if len(np.where(niva_data > y0_v + 0.10 * A_v)[0]) > 0 else tid_data[0]
        t85 = tid_data[np.where(niva_data > y0_v + 0.85 * A_v)[0][0]] if len(np.where(niva_data > y0_v + 0.85 * A_v)[0]) > 0 else tid_data[-1]
        t63 = tid_data[np.where(niva_data > y0_v + 0.63 * A_v)[0][0]] if len(np.where(niva_data > y0_v + 0.63 * A_v)[0]) > 0 else tid_data[-1]

        L_est = max(0, float(t10 - 0.05 * (t85 - t10)))
        T_est = max(0.1, float(t63 - L_est))

        # === 3. PLOTT-FUNKSJON (Beholdt alle linjer og verdier) ===
        plot_out = Output(layout={'width': '100%', 'max_width': '650px'})

        def update_plot(change=None):
            with plot_out:
                clear_output(wait=True)
                A, T, L, y0 = A_s.value, T_s.value, L_s.value, y0_s.value
                y_model = np.where(tid_data < L, y0, y0 + A * (1 - np.exp(-(tid_data - L) / T)))

                fig, ax = plt.subplots(figsize=(8, 5))
                ax.plot(tid_data, niva_data, "b.", markersize=3, alpha=0.3, label="Måledata")
                ax.plot(tid_data, y_model, "r-", linewidth=2, label="FOPDT Modell")

                ax.axhline(y0, color='black', linestyle='--', alpha=0.4)
                ax.text(tid_data[0]+15, y0, f' y0={y0:.1f}', fontweight='bold', va='bottom')
                ax.axvline(L, color='orange', linestyle=':', linewidth=2)
                ax.text(L+90, y0+30, f' L={L:.1f}s ', color='orange', fontweight='bold', ha='right')

                y63 = y0 + 0.63 * A
                ax.axhline(y63, color='green', linestyle=':', alpha=0.4)
                ax.axvline(L+T, color='green', linestyle=':', alpha=0.4)
                ax.plot(L+T, y63, 'go', markersize=8)
                ax.text(L+T, y63, f' y63={y63:.1f}\n T={T:.1f}s', color='green', fontweight='bold', va='top', ha='left')

                ax.vlines(tid_data[-1], y0, y0+A, color='purple', linewidth=3)
                ax.text(tid_data[-1], y0+A/2, f' Δy={A:.1f}', color='purple', fontweight='bold', ha='left')

                ax.grid(True, alpha=0.2); ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå")
                ax.legend(loc='lower right', fontsize='small')
                plt.tight_layout(); plt.show()

        # === 4. MOBIL-SLIDERE (Beholdt Label over slider) ===
        def create_mobile_slider(val, v_min, v_max, step, desc):
            s = FloatSlider(value=val, min=v_min, max=v_max, step=step,
                            layout={'width': '100%', 'height': '40px'},
                            continuous_update=True)
            l = Label(value=desc)
            return s, VBox([l, s], layout={'width': '100%', 'padding': '5px 0'})

        A_s, A_box = create_mobile_slider(A_v, A_v*0.2, A_v*2, 0.01, f"Forsterkning (Δy):")
        T_s, T_box = create_mobile_slider(T_est, 0.1, T_est*4, 0.1, f"Tidskonstant (T):")
        L_s, L_box = create_mobile_slider(L_est, 0, tid_data[-1]/2, 0.1, f"Dødtid (L):")
        y0_s, y0_box = create_mobile_slider(y0_v, y0_v-10, y0_v+10, 0.01, f"Startnivå (y0):")

        for s in [A_s, T_s, L_s, y0_s]: s.observe(update_plot, "value")

        # === 5. INFO-BOKS (Beholdt tabell, endret formler til ren HTML) ===
        table_html = """
        <div style="font-family: sans-serif; border: 1px solid #ddd; border-radius: 8px; background: #fff; padding: 10px; margin-top: 10px;">
            <h4 style="text-align: center; margin: 0 0 10px 0;">SIMC Regulering</h4>
            <table style="width: 100%; border-collapse: collapse; table-layout: fixed; font-size: 0.85em;">
                <tr style="background: #f4f4f4;">
                    <th style="width:33%; border: 1px solid #eee; padding: 5px;">λ</th>
                    <th style="width:33%; border: 1px solid #eee; padding: 5px;">Respons</th>
                    <th style="width:33%; border: 1px solid #eee; padding: 5px;">Obs.</th>
                </tr>
                <tr><td style="border: 1px solid #eee; text-align: center; padding: 5px;">T/2</td><td style="border: 1px solid #eee; text-align: center;">Rolig</td><td style="border: 1px solid #eee; text-align: center;">Robust</td></tr>
                <tr><td style="border: 1px solid #eee; text-align: center; padding: 5px;">T/4</td><td style="border: 1px solid #eee; text-align: center;">Std.</td><td style="border: 1px solid #eee; text-align: center;">Balanse</td></tr>
                <tr><td style="border: 1px solid #eee; text-align: center; padding: 5px;">T/6</td><td style="border: 1px solid #eee; text-align: center;">Rask</td><td style="border: 1px solid #eee; text-align: center;">Aggressiv</td></tr>
            </table>
            <div style="background: #f9f9f9; padding: 10px; border-radius: 8px; border: 1px solid #eee; margin-top: 15px; font-size: 0.95em; line-height: 1.6;">
                <b style="color: #333;">PID Formler (PI):</b><br>
                • <b>K</b> = Δy / Δu <br>
                • <b>Kp</b> = T / (K · (λ + L)) <br>
                • <b>Ti</b> = min(T, 4 · (λ + L))
            </div>
        </div>
        """

        kontroller = VBox([A_box, T_box, L_box, y0_box], layout={'flex': '1 1 300px', 'width': '100%', 'padding': '10px'})
        info_kolonne = VBox([widgets.HTML(table_html)], layout={'flex': '1 1 320px', 'width': '100%', 'padding': '10px'})
        dashbord = HBox([plot_out, kontroller, info_kolonne], layout={'flex_flow': 'row wrap', 'width': '100%', 'align_items': 'flex-start'})

        display(dashbord)
        update_plot()

# === Start programmet ===
uploader.observe(start_analysen, names='value')
display(Markdown("# FOPDT Simulator"), uploader, app_display)
